# 🔧 DEV Environment Pipeline
**Branch:** `develop` | **DB:** `DEV_DB` | **Triggers:** every push to `develop`

### What this notebook does
1. Validate DEV_DB environment is healthy
2. Run schema migrations from `migrations/` folder
3. Seed sample/test data
4. Run unit tests
5. Log deployment result

In [ ]:
# ── CELL 1: Init
import time
from snowflake.snowpark.context import get_active_session

session = get_active_session()
ENV     = 'DEV'
DB      = 'DEV_DB'
start   = time.time()
results = {}

print(f'🔧 {ENV} Pipeline Starting')
print(f'   DB      : {session.get_current_database()}')
print(f'   Role    : {session.get_current_role()}')
print(f'   WH      : {session.get_current_warehouse()}')

In [ ]:
# ── CELL 2: Environment Health Check
print('Checking DEV_DB health...')

checks = [
    ('Database reachable',  f"SELECT COUNT(*) FROM {DB}.INFORMATION_SCHEMA.TABLES"),
    ('RAW schema exists',   f"SELECT COUNT(*) FROM {DB}.INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME='RAW'"),
    ('STAGING exists',      f"SELECT COUNT(*) FROM {DB}.INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME='STAGING'"),
    ('ANALYTICS exists',    f"SELECT COUNT(*) FROM {DB}.INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME='ANALYTICS'"),
]

for name, sql in checks:
    val = session.sql(sql).collect()[0][0]
    ok  = val > 0
    print(f"  {'✓' if ok else '✗'} {name}: {val}")

results['health'] = 'OK'
print('Health check passed.')

In [ ]:
# ── CELL 3: Run Migrations from Git repo
print('Running migrations...')

migration_files = session.sql("""
    SELECT RELATIVE_PATH FROM DIRECTORY(
        @DEV_DB.ANALYTICS.cicd_stage
    )
    WHERE RELATIVE_PATH LIKE 'migrations/%.sql'
    ORDER BY RELATIVE_PATH
""").collect()

ran = 0
for row in migration_files:
    path = row[0]
    try:
        sql_rows = session.sql(f"SELECT $1 FROM @DEV_DB.ANALYTICS.cicd_stage/{path}").collect()
        sql_text = ' '.join(r[0] for r in sql_rows)
        for stmt in sql_text.split(';'):
            stmt = stmt.strip()
            if stmt and not stmt.startswith('--'):
                # Run in DEV_DB context
                session.sql(stmt.replace('DEMO_DB', 'DEV_DB')).collect()
        ran += 1
        print(f'  ✓ {path}')
    except Exception as e:
        print(f'  ~ {path}: {e}')

results['migrations'] = f'{ran} files'
print(f'Migrations: {ran} applied')

In [ ]:
# ── CELL 4: Seed Sample Data (DEV only)
print('Seeding sample data for DEV testing...')

session.sql("USE DATABASE DEV_DB").collect()

# Create sample tables if not exist
session.sql("""
    CREATE TABLE IF NOT EXISTS DEV_DB.RAW.CUSTOMERS (
        customer_id NUMBER, name VARCHAR(200),
        email VARCHAR(200), region VARCHAR(50),
        created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )
""").collect()

session.sql("""
    CREATE TABLE IF NOT EXISTS DEV_DB.RAW.ORDERS (
        order_id NUMBER, customer_id NUMBER,
        amount FLOAT, status VARCHAR(50),
        order_date DATE, region VARCHAR(50),
        created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )
""").collect()

# Seed 10 sample customers
session.sql("DELETE FROM DEV_DB.RAW.CUSTOMERS WHERE customer_id <= 10").collect()
session.sql("""
    INSERT INTO DEV_DB.RAW.CUSTOMERS (customer_id, name, email, region)
    SELECT seq4()+1, 'Customer_'||seq4(), 'cust'||seq4()||'@demo.com',
           CASE MOD(seq4(),4) WHEN 0 THEN 'NORTH' WHEN 1 THEN 'SOUTH'
           WHEN 2 THEN 'EAST' ELSE 'WEST' END
    FROM TABLE(GENERATOR(ROWCOUNT=>10))
""").collect()

# Seed 50 sample orders
session.sql("DELETE FROM DEV_DB.RAW.ORDERS WHERE order_id <= 50").collect()
session.sql("""
    INSERT INTO DEV_DB.RAW.ORDERS (order_id, customer_id, amount, status, order_date, region)
    SELECT seq4()+1, MOD(seq4(),10)+1, UNIFORM(10,1000,RANDOM())::FLOAT,
           CASE MOD(seq4(),3) WHEN 0 THEN 'completed' WHEN 1 THEN 'pending' ELSE 'cancelled' END,
           DATEADD('day', -MOD(seq4(),90), CURRENT_DATE()),
           CASE MOD(seq4(),4) WHEN 0 THEN 'NORTH' WHEN 1 THEN 'SOUTH'
           WHEN 2 THEN 'EAST' ELSE 'WEST' END
    FROM TABLE(GENERATOR(ROWCOUNT=>50))
""").collect()

c_count = session.sql("SELECT COUNT(*) FROM DEV_DB.RAW.CUSTOMERS").collect()[0][0]
o_count = session.sql("SELECT COUNT(*) FROM DEV_DB.RAW.ORDERS").collect()[0][0]
print(f'  ✓ Customers: {c_count} rows')
print(f'  ✓ Orders   : {o_count} rows')
results['seed'] = 'OK'

In [ ]:
# ── CELL 5: Unit Tests
print('Running DEV unit tests...')

tests = [
    ('Customers not empty',    'SELECT COUNT(*) FROM DEV_DB.RAW.CUSTOMERS', lambda v: v > 0),
    ('Orders not empty',       'SELECT COUNT(*) FROM DEV_DB.RAW.ORDERS', lambda v: v > 0),
    ('No null customer_id',    'SELECT COUNT(*) FROM DEV_DB.RAW.ORDERS WHERE customer_id IS NULL', lambda v: v == 0),
    ('No negative amounts',    'SELECT COUNT(*) FROM DEV_DB.RAW.ORDERS WHERE amount < 0', lambda v: v == 0),
    ('Valid status values',    "SELECT COUNT(*) FROM DEV_DB.RAW.ORDERS WHERE status NOT IN ('completed','pending','cancelled')", lambda v: v == 0),
    ('Valid regions',          "SELECT COUNT(*) FROM DEV_DB.RAW.ORDERS WHERE region NOT IN ('NORTH','SOUTH','EAST','WEST')", lambda v: v == 0),
]

passed = failed = 0
for name, sql, check in tests:
    val = session.sql(sql).collect()[0][0]
    ok  = check(val)
    print(f"  {'✓' if ok else '✗'} {name}")
    passed += ok; failed += not ok

results['tests'] = f'{passed}/{passed+failed} passed'
print(f'\nTests: {passed} passed, {failed} failed')
if failed > 0:
    raise Exception(f'DEV tests failed: {failed} failures')

In [ ]:
# ── CELL 6: Log Deployment
import json

duration = round(time.time() - start, 1)
results['duration_sec'] = duration

session.sql(f"""
    INSERT INTO DEV_DB.ANALYTICS.DEPLOY_LOG
        (environment, branch, status, duration_sec, notes)
    VALUES ('DEV', 'develop', 'SUCCESS', {duration}, '{json.dumps(results)}')
""").collect()

print()
print('=' * 50)
print(f'  🔧 DEV DEPLOY COMPLETE  ({duration}s)')
print('=' * 50)
for k, v in results.items():
    print(f'  {k:20s}: {v}')
print()
print('  Next: create a PR from develop → uat')